In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import openpyxl
from datetime import datetime as dt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def get_yearly_price_data(ticker, year):
    """
    Get opening price, closing price, and average volume for a specific year.
    
    Args:
        ticker: yfinance Ticker object
        year: Year to analyze (int)
    
    Returns:
        dict: Contains opening_price, closing_price, avg_volume
    """
    try:
        # Get historical data for the specific year
        start_date = f"{year}-01-01"
        end_date = f"{year+1}-01-01"
        
        hist = ticker.history(start=start_date, end=end_date)
        
        if hist.empty:
            return {
                'opening_price': np.nan,
                'closing_price': np.nan,
                'avg_volume': np.nan
            }
        
        # Get first and last trading day prices
        opening_price = hist['Open'].iloc[0]
        closing_price = hist['Close'].iloc[-1]
        avg_volume = hist['Volume'].mean()
        
        return {
            'opening_price': opening_price,
            'closing_price': closing_price,
            'avg_volume': avg_volume
        }
    
    except Exception as e:
        print(f"Error getting price data for {year}: {e}")
        return {
            'opening_price': np.nan,
            'closing_price': np.nan,
            'avg_volume': np.nan
        }

def calculate_ratios(data_dict, year):
    """
    Calculate derived metrics and ratios for a given year.
    
    Args:
        data_dict: Dictionary containing financial data for the year
        year: Year being processed
    
    Returns:
        dict: Updated dictionary with calculated ratios
    """
    try:
        # Calculate margins
        revenue = data_dict.get('Revenue', np.nan)
        if pd.notna(revenue) and revenue != 0:
            data_dict['Gross Margin %'] = (data_dict.get('Gross Profit', 0) / revenue) * 100
            data_dict['EBITDA Margin %'] = (data_dict.get('EBITDA', 0) / revenue) * 100
            data_dict['EBIT Margin %'] = (data_dict.get('EBIT', 0) / revenue) * 100
            data_dict['Net Margin %'] = (data_dict.get('Net Income', 0) / revenue) * 100
        else:
            data_dict['Gross Margin %'] = np.nan
            data_dict['EBITDA Margin %'] = np.nan
            data_dict['EBIT Margin %'] = np.nan
            data_dict['Net Margin %'] = np.nan
        
        # Calculate Debt-to-Equity
        total_debt = (data_dict.get('Short Term Debt', 0) or 0) + (data_dict.get('Long Term Debt', 0) or 0)
        total_equity = data_dict.get('Total Equity', np.nan)
        if pd.notna(total_equity) and total_equity != 0:
            data_dict['Debt to Equity'] = total_debt / total_equity
        else:
            data_dict['Debt to Equity'] = np.nan
        
        # Calculate per-share metrics
        shares = data_dict.get('Shares Outstanding', np.nan)
        if pd.notna(shares) and shares != 0:
            dividend_payment = abs(data_dict.get('Dividend Payment', 0) or 0)  # Make positive
            data_dict['Dividend Per Share'] = dividend_payment / shares
        else:
            data_dict['Dividend Per Share'] = np.nan
        
        # Calculate PE Ratio using year-end price and EPS
        closing_price = data_dict.get('Closing Price', np.nan)
        eps = data_dict.get('EPS', np.nan)
        if pd.notna(closing_price) and pd.notna(eps) and eps != 0:
            data_dict['PE Ratio (Year-End Price)'] = closing_price / eps
        else:
            data_dict['PE Ratio (Year-End Price)'] = np.nan
        
        # Calculate Dividend Yield
        dps = data_dict.get('Dividend Per Share', np.nan)
        if pd.notna(closing_price) and pd.notna(dps) and closing_price != 0:
            data_dict['Dividend Yield %'] = (dps / closing_price) * 100
        else:
            data_dict['Dividend Yield %'] = np.nan
        
        # Calculate Payout Ratio
        if pd.notna(eps) and pd.notna(dps) and eps != 0:
            data_dict['Payout Ratio %'] = (dps / eps) * 100
        else:
            data_dict['Payout Ratio %'] = np.nan
        
        return data_dict
    
    except Exception as e:
        print(f"Error calculating ratios for {year}: {e}")
        return data_dict

In [ ]:
def get_single_stock_analysis(ticker_symbol, years_back=10):
    """
    Comprehensive financial analysis of a single stock over specified years.
    
    Args:
        ticker_symbol: Stock ticker (e.g., 'MSFT')
        years_back: Number of years to analyze (default 10)
    
    Returns:
        dict: Complete financial analysis data
    """
    print(f"Analyzing {ticker_symbol}...")
    
    try:
        ticker = yf.Ticker(ticker_symbol)
        info = ticker.info
        
        # Basic company information
        company_info = {
            'Company Name': info.get('longName', ticker_symbol),
            'Ticker': ticker_symbol,
            'Current Price': info.get('currentPrice', np.nan),
            'Currency': info.get('currency', 'USD'),
            'Exchange': info.get('fullExchangeName', 'Unknown')
        }
        
        print(f"Company: {company_info['Company Name']}")
        
        # Get financial statements
        financials = ticker.financials
        balance_sheet = ticker.balance_sheet
        cashflow = ticker.cashflow
        
        # Determine available years
        current_year = dt.now().year
        target_years = list(range(current_year - years_back + 1, current_year + 1))
        
        # Get all available years from financial statements
        available_years = set()
        if not financials.empty:
            available_years.update(financials.columns.year)
        if not balance_sheet.empty:
            available_years.update(balance_sheet.columns.year)
        if not cashflow.empty:
            available_years.update(cashflow.columns.year)
        
        # Filter to target years that have data
        analysis_years = [year for year in target_years if year in available_years]
        analysis_years.sort()
        
        print(f"Analyzing years: {analysis_years}")
        
        yearly_data = {}
        
        for year in analysis_years:
            print(f"Processing {year}...")
            year_data = {'Year': year}
            
            # Helper function to get data for specific year
            def get_financial_data(df, item_name, year):
                if df.empty or item_name not in df.index:
                    return np.nan
                year_cols = [col for col in df.columns if col.year == year]
                if not year_cols:
                    return np.nan
                return df.loc[item_name, year_cols[0]]
            
            # P&L Statement data
            year_data['Revenue'] = get_financial_data(financials, 'Total Revenue', year)
            year_data['Gross Profit'] = get_financial_data(financials, 'Gross Profit', year)
            year_data['EBITDA'] = get_financial_data(financials, 'EBITDA', year)
            year_data['EBIT'] = get_financial_data(financials, 'EBIT', year)
            
            # Try different names for EBT (Earnings Before Tax)
            ebt = get_financial_data(financials, 'Pretax Income', year)
            if pd.isna(ebt):
                ebt = get_financial_data(financials, 'Income Before Tax', year)
            year_data['EBT'] = ebt
            
            year_data['Net Income'] = get_financial_data(financials, 'Net Income', year)
            year_data['EPS'] = get_financial_data(financials, 'Diluted EPS', year)
            year_data['Shares Outstanding'] = get_financial_data(financials, 'Basic Average Shares', year)
            
            # Cash Flow data
            year_data['Operating Cash Flow'] = get_financial_data(cashflow, 'Operating Cash Flow', year)
            year_data['Investing Cash Flow'] = get_financial_data(cashflow, 'Investing Cash Flow', year)
            year_data['Financing Cash Flow'] = get_financial_data(cashflow, 'Financing Cash Flow', year)
            year_data['Dividend Payment'] = get_financial_data(cashflow, 'Cash Dividends Paid', year)
            
            # Balance Sheet data
            total_assets = get_financial_data(balance_sheet, 'Total Assets', year)
            current_assets = get_financial_data(balance_sheet, 'Current Assets', year)
            
            year_data['Current Assets'] = current_assets
            year_data['Fixed Assets'] = total_assets - current_assets if pd.notna(total_assets) and pd.notna(current_assets) else np.nan
            
            # Try different debt naming conventions
            short_debt = get_financial_data(balance_sheet, 'Current Debt', year)
            if pd.isna(short_debt):
                short_debt = get_financial_data(balance_sheet, 'Current Debt And Capital Lease Obligation', year)
            year_data['Short Term Debt'] = short_debt
            
            year_data['Long Term Debt'] = get_financial_data(balance_sheet, 'Long Term Debt', year)
            
            # Try different equity naming conventions
            total_equity = get_financial_data(balance_sheet, 'Total Equity Gross Minority Interest', year)
            if pd.isna(total_equity):
                total_equity = get_financial_data(balance_sheet, 'Stockholders Equity', year)
            year_data['Total Equity'] = total_equity
            
            # Get trading information
            price_data = get_yearly_price_data(ticker, year)
            year_data['Opening Price'] = price_data['opening_price']
            year_data['Closing Price'] = price_data['closing_price']
            year_data['Average Volume'] = price_data['avg_volume']
            
            # Calculate ratios and derived metrics
            year_data = calculate_ratios(year_data, year)
            
            # Calculate PE Ratio using current price and historical EPS
            current_price = company_info.get('Current Price', np.nan)
            eps = year_data.get('EPS', np.nan)
            if pd.notna(current_price) and pd.notna(eps) and eps != 0:
                year_data['PE Ratio (Current Price)'] = current_price / eps
            else:
                year_data['PE Ratio (Current Price)'] = np.nan
            
            yearly_data[year] = year_data
        
        result = {
            'company_info': company_info,
            'yearly_data': yearly_data,
            'analysis_years': analysis_years
        }
        
        print(f"Analysis complete for {ticker_symbol}")
        return result
    
    except Exception as e:
        print(f"Error analyzing {ticker_symbol}: {e}")
        return None

In [ ]:
def export_to_excel(analysis_data, output_filename):
    """
    Export single stock analysis to Excel with proper formatting.
    
    Args:
        analysis_data: Result from get_single_stock_analysis()
        output_filename: Path for Excel file
    """
    if not analysis_data:
        print("No data to export")
        return
    
    try:
        company_info = analysis_data['company_info']
        yearly_data = analysis_data['yearly_data']
        analysis_years = analysis_data['analysis_years']
        
        # Define the structure of metrics in order
        metric_groups = {
            'Company Information': {
                'Company Name': 'company_info',
                'Ticker': 'company_info',
                'Current Price': 'company_info',
                'Currency': 'company_info',
                'Exchange': 'company_info'
            },
            'P&L Statement': {
                'Revenue': 'yearly',
                'Gross Profit': 'yearly',
                'Gross Margin %': 'yearly',
                'EBITDA': 'yearly',
                'EBITDA Margin %': 'yearly',
                'EBIT': 'yearly',
                'EBIT Margin %': 'yearly',
                'EBT': 'yearly',
                'Net Income': 'yearly',
                'Net Margin %': 'yearly',
                'EPS': 'yearly'
            },
            'Cash Flow Statement': {
                'Operating Cash Flow': 'yearly',
                'Investing Cash Flow': 'yearly',
                'Financing Cash Flow': 'yearly',
                'Dividend Payment': 'yearly'
            },
            'Balance Sheet': {
                'Current Assets': 'yearly',
                'Fixed Assets': 'yearly',
                'Short Term Debt': 'yearly',
                'Long Term Debt': 'yearly',
                'Total Equity': 'yearly',
                'Debt to Equity': 'yearly'
            },
            'Trading Information': {
                'Opening Price': 'yearly',
                'Closing Price': 'yearly',
                'Average Volume': 'yearly',
                'Shares Outstanding': 'yearly',
                'PE Ratio (Year-End Price)': 'yearly',
                'PE Ratio (Current Price)': 'yearly',
                'Dividend Per Share': 'yearly',
                'Dividend Yield %': 'yearly',
                'Payout Ratio %': 'yearly'
            }
        }
        
        # Prepare data for DataFrame
        data_for_df = []
        
        for group_name, metrics in metric_groups.items():
            for metric_name, data_type in metrics.items():
                row = {'Category': group_name, 'Metric': metric_name}
                
                if data_type == 'company_info':
                    # Same value for all years for company info
                    value = company_info.get(metric_name, '')
                    for year in analysis_years:
                        row[str(year)] = value
                else:
                    # Yearly data
                    for year in analysis_years:
                        if year in yearly_data:
                            row[str(year)] = yearly_data[year].get(metric_name, np.nan)
                        else:
                            row[str(year)] = np.nan
                
                data_for_df.append(row)
        
        # Create DataFrame
        df = pd.DataFrame(data_for_df)
        
        # Set multi-index for better organization
        df.set_index(['Category', 'Metric'], inplace=True)
        
        # Export to Excel
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Financial Analysis')
            
            # Get workbook and worksheet for formatting
            workbook = writer.book
            worksheet = writer.sheets['Financial Analysis']
            
            # Auto-adjust column widths
            for column in worksheet.columns:
                max_length = 0
                column = [cell for cell in column]
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column[0].column_letter].width = adjusted_width
        
        print(f"Excel file exported successfully: {output_filename}")
        
    except Exception as e:
        print(f"Error exporting to Excel: {e}")

In [11]:
# Example usage
ticker_to_analyze = "MSFT"  # Change this to any ticker you want to analyze
years_to_analyze = 10       # Number of years back to analyze

# Perform the analysis
result = get_single_stock_analysis(ticker_to_analyze, years_to_analyze)

if result:
    # Generate filename with timestamp
    today = dt.today().strftime('%Y%m%d_%H%M')
    filename = f"C:\\Users\\julia\\Downloads\\{ticker_to_analyze}_financial_analysis_{today}.xlsx"
    
    # Export to Excel
    export_to_excel(result, filename)
    
    print(f"\nAnalysis complete for {ticker_to_analyze}")
    print(f"Excel file saved: {filename}")
else:
    print(f"Failed to analyze {ticker_to_analyze}")

Analyzing MSFT...
Company: Microsoft Corporation
Analyzing years: [2021, 2022, 2023, 2024, 2025]
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...
Analysis complete for MSFT
Excel file exported successfully: C:\Users\julia\Downloads\MSFT_financial_analysis_20250820_1307.xlsx

Analysis complete for MSFT
Excel file saved: C:\Users\julia\Downloads\MSFT_financial_analysis_20250820_1307.xlsx
